# 대중교통 x 택시 수단분담률 분석
- **내부 데이터**: D012 (요금정보) - 승차건수, 시간대, 행정동코드
- **외부 데이터**: 서울 지하철 역별 시간대별 승하차 인원
- **분석 목표**: 택시 vs 지하철 수단분담률 시계열, 막차 후 택시 스파이크, 호선별 대체관계

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# === 메모리 최적화 유틸 ===
import gc, psutil, os

def mem_usage():
    """현재 RAM 사용량 출력"""
    gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'RAM: {gb:.1f} GB')

def optimize_dtypes(df, cat_cols=None):
    """DataFrame 메모리 최적화"""
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    if cat_cols:
        for col in cat_cols:
            if col in df.columns:
                df[col] = df[col].astype('category')
    return df

CHUNK_SIZE = 1_000_000  # D012 chunk 크기
D012_CAT_COLS = ['RIDE_A_CD', 'ALIGHT_A_CD', 'DRIVER_ID', 'TAXI_VEHC_ID', 'TRANSP_BIZR_ID']

mem_usage()

## 1. 데이터 로드

In [ ]:
# === 경로 설정 ===
D012_PATH = '../DC_TBYXD012.csv'
SUBWAY_PATH = '../external_data/transit/seoul_subway_hourly_ridership_all.csv'

# 지하철 데이터 로드
subway = pd.read_csv(SUBWAY_PATH, encoding='utf-8-sig')
print(f"지하철 데이터: {len(subway):,}행")
print(f"컬럼: {list(subway.columns[:5])} ... ({len(subway.columns)}개)")
print(f"기간: {subway['사용월'].min()} ~ {subway['사용월'].max()}")
subway.head(2)

In [ ]:
# 지하철 데이터를 시간대별 long format으로 변환
hours_cols = []
for col in subway.columns:
    if '승차인원' in col or '하차인원' in col:
        hours_cols.append(col)

# 시간대별 총 승차인원 집계 (전체 역 합산)
ride_cols = [c for c in subway.columns if '승차인원' in c]
alight_cols = [c for c in subway.columns if '하차인원' in c]

# 시간대 추출 함수
def extract_hour(col_name):
    """'04시-05시 승차인원' -> 4"""
    return int(col_name.split('시')[0])

# 월별 시간대별 지하철 총 승차인원
subway_hourly_rows = []
for _, row in subway.iterrows():
    month = row['사용월']
    line = row['호선명']
    station = row['지하철역']
    for col in ride_cols:
        hour = extract_hour(col)
        rides = row[col]
        subway_hourly_rows.append({'month': month, 'line': line, 'station': station, 
                                    'hour': hour, 'subway_rides': rides})

subway_hourly = pd.DataFrame(subway_hourly_rows)
print(f"변환 완료: {len(subway_hourly):,}행")
subway_hourly.head()

In [ ]:
# D012 chunk 로드 (메모리 최적화)
d012_cols = ['RIDE_DTIME', 'PAY_AMT', 'RIDE_DIST', 'RIDE_A_CD', 'ALIGHT_A_CD']
d012_dtypes = {'RIDE_DTIME': str, 'PAY_AMT': 'int32', 'RIDE_DIST': 'int32'}

hourly_list = []
monthly_list = []
dow_list = []

for chunk in pd.read_csv(D012_PATH, usecols=d012_cols, dtype=d012_dtypes, chunksize=CHUNK_SIZE):
    chunk['ride_datetime'] = pd.to_datetime(chunk['RIDE_DTIME'], format='%Y%m%d%H%M%S', errors='coerce')
    chunk = chunk.dropna(subset=['ride_datetime'])
    chunk['hour'] = chunk['ride_datetime'].dt.hour
    chunk['year_month'] = chunk['ride_datetime'].dt.to_period('M').astype(str).str.replace('-', '').astype(int)
    chunk['date'] = chunk['ride_datetime'].dt.date
    chunk['weekday'] = chunk['ride_datetime'].dt.dayofweek
    
    hourly_list.append(chunk.groupby('hour').size().reset_index(name='count'))
    monthly_list.append(chunk.groupby('year_month').size().reset_index(name='count'))
    dow_list.append(chunk.groupby(['weekday', 'hour']).size().reset_index(name='count'))
    
    n_dates = chunk['date'].nunique()
    del chunk
    gc.collect()

# 합산
taxi_hourly = pd.concat(hourly_list).groupby('hour')['count'].sum().reset_index()
taxi_hourly.columns = ['hour', 'taxi_rides']
taxi_monthly = pd.concat(monthly_list).groupby('year_month')['count'].sum().reset_index()
taxi_monthly.columns = ['year_month', 'taxi_rides']
taxi_dow = pd.concat(dow_list).groupby(['weekday', 'hour'])['count'].sum().reset_index()
taxi_dow.columns = ['weekday', 'hour', 'taxi_rides']

del hourly_list, monthly_list, dow_list
gc.collect()

# 월수 계산
n_months_taxi = taxi_monthly['year_month'].nunique()
taxi_hourly['taxi_rides_monthly'] = taxi_hourly['taxi_rides'] / n_months_taxi

print(f"택시 시간대별 집계 완료, 월수: {n_months_taxi}")
mem_usage()

## 2. 시간대별 택시 vs 지하철 수요 비교

In [ ]:
# 택시: 시간대별 월평균 승차건수
taxi_hourly = d012.groupby('hour').agg(
    taxi_rides=('PAY_AMT', 'count')
).reset_index()
# 월수로 나눠서 월평균
n_months_taxi = d012['year_month'].nunique()
taxi_hourly['taxi_rides_monthly'] = taxi_hourly['taxi_rides'] / n_months_taxi

# 지하철: 시간대별 월평균 승차인원
subway_hour_total = subway_hourly.groupby('hour')['subway_rides'].sum().reset_index()
n_months_subway = subway_hourly['month'].nunique()
subway_hour_total['subway_rides_monthly'] = subway_hour_total['subway_rides'] / n_months_subway

# 합치기
compare = taxi_hourly[['hour', 'taxi_rides_monthly']].merge(
    subway_hour_total[['hour', 'subway_rides_monthly']], on='hour', how='outer'
).fillna(0)

# 택시 비중 (분담률)
compare['total'] = compare['taxi_rides_monthly'] + compare['subway_rides_monthly']
compare['taxi_share_pct'] = (compare['taxi_rides_monthly'] / compare['total'] * 100).round(2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# 상: 절대량 비교 (dual axis)
ax1_twin = ax1.twinx()
ax1.bar(compare['hour'] - 0.2, compare['subway_rides_monthly'], width=0.4, 
        label='지하철 승차', color='#4CAF50', alpha=0.7)
ax1_twin.bar(compare['hour'] + 0.2, compare['taxi_rides_monthly'], width=0.4, 
             label='택시 승차', color='#FF9800', alpha=0.7)
ax1.set_xlabel('시간대')
ax1.set_ylabel('지하철 월평균 승차인원', color='#4CAF50')
ax1_twin.set_ylabel('택시 월평균 승차건수', color='#FF9800')
ax1.set_title('시간대별 지하철 vs 택시 수요 비교', fontsize=14, fontweight='bold')
ax1.set_xticks(range(24))
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# 하: 택시 분담률
colors = ['#F44336' if v > 50 else '#FF9800' if v > 20 else '#2196F3' for v in compare['taxi_share_pct']]
ax2.bar(compare['hour'], compare['taxi_share_pct'], color=colors)
ax2.set_xlabel('시간대')
ax2.set_ylabel('택시 분담률 (%)')
ax2.set_title('시간대별 택시 수단분담률 (택시 / (택시+지하철))', fontsize=14, fontweight='bold')
ax2.set_xticks(range(24))
ax2.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50%')
ax2.grid(True, alpha=0.3)

for i, v in enumerate(compare['taxi_share_pct']):
    if v > 5:
        ax2.text(compare['hour'].iloc[i], v + 1, f'{v:.0f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print("\n시간대별 택시 분담률:")
compare[['hour', 'taxi_rides_monthly', 'subway_rides_monthly', 'taxi_share_pct']]

## 3. 막차 이후 택시 수요 스파이크 분석

In [ ]:
# 심야시간대(23시~04시) 집중 분석
night_hours = [22, 23, 0, 1, 2, 3, 4]

night_taxi = taxi_hourly[taxi_hourly['hour'].isin(night_hours)].copy()
night_subway = subway_hour_total[subway_hour_total['hour'].isin(night_hours)].copy()

# 정렬을 위한 순서 부여
hour_order = {22: 0, 23: 1, 0: 2, 1: 3, 2: 4, 3: 5, 4: 6}
night_taxi['order'] = night_taxi['hour'].map(hour_order)
night_subway['order'] = night_subway['hour'].map(hour_order)
night_taxi = night_taxi.sort_values('order')
night_subway = night_subway.sort_values('order')

fig, ax = plt.subplots(figsize=(12, 6))
ax_twin = ax.twinx()

labels = ['22시', '23시', '0시', '1시', '2시', '3시', '4시']
x = range(len(labels))

ax.bar([i - 0.2 for i in x], night_subway['subway_rides_monthly'].values, width=0.4, 
       label='지하철', color='#4CAF50', alpha=0.7)
ax_twin.bar([i + 0.2 for i in x], night_taxi['taxi_rides_monthly'].values, width=0.4, 
            label='택시', color='#FF9800', alpha=0.7)

# 막차 시점 표시
ax.axvline(x=1.5, color='red', linestyle='--', linewidth=2, label='지하철 막차 (약 23:30~00:00)')

ax.set_xlabel('시간대', fontsize=12)
ax.set_ylabel('지하철 월평균 승차인원', color='#4CAF50', fontsize=12)
ax_twin.set_ylabel('택시 월평균 승차건수', color='#FF9800', fontsize=12)
ax.set_title('심야시간대 지하철 종료 → 택시 수요 전환 패턴', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend(loc='upper left')
ax_twin.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 막차 전후 택시 수요 변화
pre_midnight = taxi_hourly[taxi_hourly['hour'] == 22]['taxi_rides_monthly'].values[0]
post_midnight = taxi_hourly[taxi_hourly['hour'] == 0]['taxi_rides_monthly'].values[0]
print(f"\n22시 택시 수요: {pre_midnight:,.0f}건/월")
print(f"0시(막차 후) 택시 수요: {post_midnight:,.0f}건/월")
print(f"변화율: {((post_midnight/pre_midnight)-1)*100:+.1f}%")

## 4. 월별 수단분담률 추이

In [ ]:
# 택시 월별 승차건수
taxi_monthly = d012.groupby('year_month').size().reset_index(name='taxi_rides')

# 지하철 월별 총 승차인원
subway_monthly = subway_hourly.groupby('month')['subway_rides'].sum().reset_index()
subway_monthly.columns = ['year_month', 'subway_rides']

# 조인 (공통 기간만)
monthly_compare = taxi_monthly.merge(subway_monthly, on='year_month', how='inner')
monthly_compare['taxi_share'] = (monthly_compare['taxi_rides'] / 
                                  (monthly_compare['taxi_rides'] + monthly_compare['subway_rides']) * 100)

if len(monthly_compare) > 0:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))
    
    x_labels = monthly_compare['year_month'].astype(str)
    x = range(len(x_labels))
    
    # 상: 월별 이용량 추이
    ax1_twin = ax1.twinx()
    ax1.plot(x, monthly_compare['subway_rides'], color='#4CAF50', linewidth=2, label='지하철', marker='o', markersize=3)
    ax1_twin.plot(x, monthly_compare['taxi_rides'], color='#FF9800', linewidth=2, label='택시', marker='o', markersize=3)
    ax1.set_ylabel('지하철 승차인원', color='#4CAF50')
    ax1_twin.set_ylabel('택시 승차건수', color='#FF9800')
    ax1.set_title('월별 지하철 vs 택시 이용량 추이', fontsize=14, fontweight='bold')
    ax1.legend(loc='upper left')
    ax1_twin.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # 하: 택시 분담률 추이
    ax2.plot(x, monthly_compare['taxi_share'], color='#2196F3', linewidth=2, marker='o', markersize=4)
    ax2.fill_between(x, monthly_compare['taxi_share'], alpha=0.2, color='#2196F3')
    ax2.set_ylabel('택시 분담률 (%)')
    ax2.set_title('월별 택시 수단분담률 추이', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    tick_step = max(1, len(x) // 12)
    for ax in [ax1, ax2]:
        ax.set_xticks(list(range(0, len(x), tick_step)))
        ax.set_xticklabels([x_labels.iloc[i] for i in range(0, len(x), tick_step)], rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
else:
    print("공통 기간이 없습니다. D012와 지하철 데이터의 기간을 확인하세요.")
    print(f"D012 기간: {taxi_monthly['year_month'].min()} ~ {taxi_monthly['year_month'].max()}")
    print(f"지하철 기간: {subway_monthly['year_month'].min()} ~ {subway_monthly['year_month'].max()}")

## 5. 호선별 택시 대체 수요 분석

In [ ]:
# 호선별 승차인원 & 심야 비중
line_stats = subway_hourly.groupby('line').agg(
    total_rides=('subway_rides', 'sum'),
    station_count=('station', 'nunique')
).reset_index()

# 심야(22~04시) 비중
night_mask = subway_hourly['hour'].isin([22, 23, 0, 1, 2, 3, 4])
line_night = subway_hourly[night_mask].groupby('line')['subway_rides'].sum().reset_index()
line_night.columns = ['line', 'night_rides']

line_stats = line_stats.merge(line_night, on='line', how='left')
line_stats['night_pct'] = (line_stats['night_rides'] / line_stats['total_rides'] * 100).round(1)
line_stats = line_stats.sort_values('total_rides', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 호선별 총 승차인원
ax1.barh(line_stats['line'], line_stats['total_rides'], color='#4CAF50', alpha=0.7)
ax1.set_xlabel('총 승차인원')
ax1.set_title('호선별 총 승차인원', fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# 호선별 심야 비중
ax2.barh(line_stats['line'], line_stats['night_pct'], color='#FF9800', alpha=0.7)
ax2.set_xlabel('심야시간(22~04시) 비중 (%)')
ax2.set_title('호선별 심야시간 승차 비중', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("호선별 통계:")
line_stats[['line', 'total_rides', 'station_count', 'night_pct']].to_string(index=False)

## 6. 요일별 패턴 비교

In [ ]:
# 택시 요일별 시간대별 수요
taxi_dow = d012.groupby(['weekday', 'hour']).size().reset_index(name='taxi_rides')
n_weeks = d012['date'].nunique() / 7
taxi_dow['taxi_rides_weekly'] = taxi_dow['taxi_rides'] / n_weeks

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
dow_labels = ['월', '화', '수', '목', '금', '토', '일']
colors_dow = ['#2196F3', '#2196F3', '#2196F3', '#2196F3', '#FF9800', '#F44336', '#F44336']

# 좌: 요일별 총 택시 수요
dow_total = taxi_dow.groupby('weekday')['taxi_rides_weekly'].sum().reset_index()
axes[0].bar(dow_total['weekday'], dow_total['taxi_rides_weekly'], color=colors_dow)
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(dow_labels)
axes[0].set_ylabel('주평균 택시 승차건수')
axes[0].set_title('요일별 택시 수요', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# 우: 금/토 심야 vs 평일 심야 비교
weekday_night = taxi_dow[(taxi_dow['weekday'].isin([0,1,2,3])) & (taxi_dow['hour'].isin([22,23,0,1,2]))]
friday_night = taxi_dow[(taxi_dow['weekday'] == 4) & (taxi_dow['hour'].isin([22,23,0,1,2]))]
saturday_night = taxi_dow[(taxi_dow['weekday'] == 5) & (taxi_dow['hour'].isin([22,23,0,1,2]))]

night_compare = pd.DataFrame({
    '구분': ['평일 심야\n(월~목)', '금요일 심야', '토요일 심야'],
    '평균건수': [
        weekday_night['taxi_rides_weekly'].mean(),
        friday_night['taxi_rides_weekly'].mean(),
        saturday_night['taxi_rides_weekly'].mean()
    ]
})

axes[1].bar(night_compare['구분'], night_compare['평균건수'], 
            color=['#2196F3', '#FF9800', '#F44336'])
axes[1].set_ylabel('심야 시간당 평균 승차건수')
axes[1].set_title('평일 vs 금/토 심야 택시 수요', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for i, v in enumerate(night_compare['평균건수']):
    axes[1].text(i, v + v*0.02, f'{v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. 주요 역세권 택시-지하철 대체 분석

In [ ]:
# 주요 역별 시간대별 지하철 승차 패턴
major_stations = ['강남', '홍대입구', '서울역', '잠실', '여의도', '신림', '건대입구', '사당']

major_subway = subway_hourly[subway_hourly['station'].isin(major_stations)]
major_by_hour = major_subway.groupby(['station', 'hour'])['subway_rides'].mean().reset_index()

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, station in enumerate(major_stations):
    if idx >= len(axes):
        break
    ax = axes[idx]
    data = major_by_hour[major_by_hour['station'] == station]
    if len(data) > 0:
        ax.bar(data['hour'], data['subway_rides'], color='#4CAF50', alpha=0.7)
        ax.axvline(x=23.5, color='red', linestyle='--', alpha=0.5)
        ax.set_title(f'{station}역', fontweight='bold')
        ax.set_xticks([0, 6, 12, 18, 23])
        ax.grid(True, alpha=0.3, axis='y')
    else:
        ax.set_title(f'{station}역 (데이터 없음)')

fig.suptitle('주요 역 시간대별 지하철 승차 패턴 (빨간선=막차)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. 요약

In [ ]:
print("=" * 70)
print("대중교통 x 택시 수단분담률 분석 요약")
print("=" * 70)

# 시간대별 핵심 지표
peak_taxi_hour = compare.loc[compare['taxi_share_pct'].idxmax()]
min_taxi_hour = compare.loc[compare['taxi_share_pct'].idxmin()]

print(f"\n1. 택시 분담률 최고 시간대: {int(peak_taxi_hour['hour'])}시 ({peak_taxi_hour['taxi_share_pct']:.1f}%)")
print(f"2. 택시 분담률 최저 시간대: {int(min_taxi_hour['hour'])}시 ({min_taxi_hour['taxi_share_pct']:.1f}%)")
print(f"3. 전체 평균 택시 분담률: {compare['taxi_share_pct'].mean():.1f}%")

# 심야 vs 주간
daytime = compare[compare['hour'].between(6, 22)]['taxi_share_pct'].mean()
nighttime = compare[compare['hour'].isin([23, 0, 1, 2, 3, 4, 5])]['taxi_share_pct'].mean()
print(f"\n4. 주간(06~22시) 평균 택시 분담률: {daytime:.1f}%")
print(f"5. 심야(23~05시) 평균 택시 분담률: {nighttime:.1f}%")
print(f"   → 심야 택시 분담률이 주간 대비 {nighttime/daytime:.1f}배")